In [ ]:
from src.models.cvmodels import FeatureBasedEstimator
from src.models.feature_extractors import RapidTextExtractor, PaddleTextExtractor, OrbFeatureExtractor, SIFTFeatureExtractor
from src.models.deepmodels import SiameseDino
from src.feature_stores import IndexManager, TextCSRIndex, InMemoryStore
from src.distances import BinaryJaccardKernel, FeatureBasedDistance
from src.eval import Recall
from transformers import AutoImageProcessor, AutoModel
from hydra import initialize, compose
import torch

In [2]:
from pathlib import Path
from src.utils import DataPreparator, Browser
from src.data import make_transform, LazyLoadCollection
from torch.utils.data import DataLoader

dataset_path = "../data/winestickers/mini"

EXPERIMENT_DATA_PATH = Path("../data/augmented_data16")
ORIGINAL_DATA_PATH = Path(dataset_path)

data_preparator = DataPreparator(ORIGINAL_DATA_PATH, EXPERIMENT_DATA_PATH, 42)

training_size = 0.7
data_splits = data_preparator.train_val_test_split(train_ratio=0, val_ratio=1, n_query=1)
train_paths, train_labels = data_splits["train"]
gallery_paths, gallery_labels = data_splits["gallery"]
val_query_paths, val_query_labels = data_splits["val_query"]
test_query_paths, test_query_labels = data_splits["test_query"]
print(gallery_paths)
print(val_query_paths)

browser = Browser(ORIGINAL_DATA_PATH)
gallery_paths, gallery_labels, query_paths, query_labels = browser.sample_leave_k_out(1)

RESIZE_SIZE = 224

print("Loading dataloaders...")
gallery_dataset = LazyLoadCollection(gallery_paths, gallery_labels, transform=make_transform(RESIZE_SIZE))
gallery_dataloader = DataLoader(gallery_dataset, batch_size=32)

val_dataset = LazyLoadCollection(query_paths, query_labels, transform=make_transform(RESIZE_SIZE))
val_dataloader = DataLoader(val_dataset, batch_size=32)

Found 10 total classes.
Processing Train classes...
Processing Validation classes...
Processing Test classes...

--- Data Split Summary ---
Training Loader:      16 samples from 1 classes (Augmented)
Gallery Loader:       21 samples from 10 classes (Original)
Val Query Loader:      9 samples from 9 classes (Original)
Test Query Loader:     0 samples from 0 classes (Original)
['../data/winestickers/mini/009/2026-01-22_18.52.39.jpg', '../data/winestickers/mini/009/2026-01-22_18.52.33.jpg', '../data/winestickers/mini/009/2026-01-22_18.52.47.jpg', '../data/winestickers/mini/010/2026-01-22_18.53.04.jpg', '../data/winestickers/mini/010/2026-01-22_18.52.52.jpg', '../data/winestickers/mini/006/2026-01-22_18.50.43.jpg', '../data/winestickers/mini/006/2026-01-22_18.51.09.jpg', '../data/winestickers/mini/002/2026-01-22_18.50.20.jpg', '../data/winestickers/mini/002/2026-01-22_18.50.26.jpg', '../data/winestickers/mini/004/2026-01-22_18.51.43.jpg', '../data/winestickers/mini/004/2026-01-22_18.51.31.

In [3]:
rapidocr = RapidTextExtractor()
memory_store = InMemoryStore()
index = [TextCSRIndex()]
index_manager = IndexManager(index)
feature_extractors = [rapidocr]
kernels = [BinaryJaccardKernel()]
distance_strategy = FeatureBasedDistance(kernels, [1])
model4 = FeatureBasedEstimator(feature_extractors, memory_store, index_manager, distance_strategy)

In [4]:
model4.evaluate(gallery_dataloader, val_dataloader, Recall())

[codecarbon WARNING @ 16:42:33] Multiple instances of codecarbon are allowed to run at the same time.


🔨 Preparing gallery (computing features)...


Computing features: 100%|██████████| 20/20 [00:22<00:00,  1.13s/it]


✅ Gallery prepared: 20 images
Computing queries features...


Computing features blocks: 100%|██████████| 1/1 [00:13<00:00, 13.39s/it]


[[0.         0.07142857 0.04761905 ... 0.03125    0.04347826 0.13580247]
 [0.11764706 0.025      0.1        ... 0.22222222 0.15686275 0.10752688]
 [0.15       0.         0.05882353 ... 0.         0.04761905 0.        ]
 ...
 [0.         0.         0.         ... 0.         0.         0.        ]
 [0.         0.         0.         ... 0.         0.         0.        ]
 [0.         0.         0.         ... 0.         0.         0.        ]]
Elapsed time for evaluate = 36.07s
Energy consumption for evaluate = 0.000657 kWh
Carbon footprint for evaluate = 0.000037 g.eq.CO2


{'recall@1': 0.1, 'recall@3': 0.2, 'recall@10': 0.9}

In [ ]:
with initialize(config_path="../config", version_base=None):
    cfg = compose(config_name="base_config")

checkpoint = torch.load("../model_checkpoints/sandy-fire-218.pth")

backbone_name = "facebook/dinov3-vitb16-pretrain-lvd1689m"
processor = AutoImageProcessor.from_pretrained(backbone_name)
backbone = AutoModel.from_pretrained(backbone_name, dtype=torch.float32)

model1 = SiameseDino(backbone, processor, cfg)
model1.load_state_dict(checkpoint)

rapidocr = RapidTextExtractor()
orb = OrbFeatureExtractor(1000)
sift = SIFTFeatureExtractor()
feature_extractors = [orb]
weights = [1]
model2 = FeatureBasedEstimator(feature_extractors, weights)
feature_extractors = [orb, rapidocr]
weights = [0.5, 0.5]
model3 = FeatureBasedEstimator(feature_extractors, weights)
feature_extractors = [sift]
model4 = FeatureBasedEstimator(feature_extractors, [1])

In [ ]:
from src.utils import ModelReport
from src.eval import Recall

metric = Recall()
modelreport = ModelReport({"sift": model4})#"siamesedino": model1, "orb": model2, "0.5_orb+0.5_rapidocr": model3, "sift": model4})
df = modelreport.generate_report("../data/winestickers/mini", "random", metric)

[codecarbon WARNING @ 22:13:36] Multiple instances of codecarbon are allowed to run at the same time.


Found 10 total classes.
Processing Train classes...
Processing Validation classes...
Processing Test classes...

--- Data Split Summary ---
Training Loader:      16 samples from 1 classes (Augmented)
Gallery Loader:       21 samples from 10 classes (Original)
Val Query Loader:      9 samples from 9 classes (Original)
Test Query Loader:     0 samples from 0 classes (Original)
['../data/winestickers/mini/009/2026-01-22_18.52.39.jpg', '../data/winestickers/mini/009/2026-01-22_18.52.33.jpg', '../data/winestickers/mini/009/2026-01-22_18.52.47.jpg', '../data/winestickers/mini/001/2026-01-22_18.50.56.jpg', '../data/winestickers/mini/001/2026-01-22_18.52.03.jpg', '../data/winestickers/mini/005/2026-01-22_18.49.55.jpg', '../data/winestickers/mini/005/2026-01-22_18.50.37.jpg', '../data/winestickers/mini/007/2026-01-22_18.51.17.jpg', '../data/winestickers/mini/007/2026-01-22_18.52.08.jpg', '../data/winestickers/mini/006/2026-01-22_18.51.03.jpg', '../data/winestickers/mini/006/2026-01-22_18.51.09.

Computing features: 100%|██████████| 20/20 [00:24<00:00,  1.21s/it]


[[['soda', 'embub93029', 'orange', 'embsemo64421', 'casino', 'emb13215c', 'pur', 'sucre', 'servir', 'tres', 'frais', '3l3', 'f69009', 'ilitre', 'casino', '42008', 'st-etlenne', 'cedex']], [['orangina', 'boisson', 'a', 'la', 'pulpe', "d'orange", 'emb.63113clermont', 'urani', 'ao', '小川', 'contenance', 'agiterlabouteille', 'servir', 'tresfrais', 'eaugazeifiee,juspulpeuxdorange(12%dejus2%depupe）.', 'autresjusdagrumessucrezestedorangeantoxygene.', 'boisson', 'conservee', 'par', 'pasteurisation', '249760000180']], [['limonade', 'aux', 'extraits', '个', 'pur', 'naturels', 'de', 'fruits', 'j', 'za', 'oore', 'r.c.64a167', 'r.m.607-64-63']], [['1litre', 'digestive', 'rafraichissante', 'castel', 'iimonade', 'cmpositaifiidiiqucel', 'q.pigot', '13110', 'bort-les-orgues', 'garnaud-angouleme']], [['freezor', 'citron', "adiluerd'aumoins6foissonvolumed'eauglacee", 'concentre', 'tenir', 'au', 'frais', 'sirop', 'de', 'citron', 'sirop', 'de', 'sucre-', 'glucose-concentre', 'de', 'jus', 'de', 'citron', '5/1

Computing features: 100%|██████████| 10/10 [00:13<00:00,  1.31s/it]


20
(10, 336)
(20, 336)
(20, 336)
(10, 20)
[[[ 0.00000000e+00  7.14285714e-02  4.76190476e-02  0.00000000e+00
    0.00000000e+00  0.00000000e+00  8.33333333e-02  0.00000000e+00
    0.00000000e+00  2.59259259e-01  3.06122449e-02  3.33333333e-02
    0.00000000e+00  3.70370370e-02  0.00000000e+00  7.40740741e-02
    8.82352941e-02  3.12500000e-02  4.34782609e-02  1.35802469e-01]
  [ 1.17647059e-01  2.50000000e-02  1.00000000e-01  3.44827586e-02
    1.60000000e-01  1.84210526e-01  4.16666667e-02  1.11111111e-01
    0.00000000e+00  2.27272727e-02  1.42857143e-01  2.00000000e-01
    5.71428571e-02  1.47058824e-01  1.38888889e-01  2.56410256e-02
    1.70731707e-01  2.22222222e-01  1.56862745e-01  1.07526882e-01]
  [ 1.50000000e-01  0.00000000e+00  5.88235294e-02  0.00000000e+00
    0.00000000e+00  7.14285714e-02  0.00000000e+00  2.50000000e-01
    0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
    0.00000000e+00  0.00000000e+00  1.81818182e-01  4.16666667e-02
    1.37931034e-01

TypeError: OCRExtractor.compute_distances() takes 2 positional arguments but 3 were given

In [ ]:
df

,evaluation time,inference time,recall@1,recall@3,recall@10,store size (mb)
siamesedino,3.170333,0.190323,0.579545,0.727273,0.840909,6576
orb,88.769850,0.700401,0.829545,0.863636,0.886364,6576
0.5_orb+0.5_rapidocr,318.283803,1.562125,0.715909,0.761364,0.806818,6576


In [2]:
img1 = "/home/timothee/Documents/programmation/DIHT/data/original_data/002/IMG_20240220_175710405.jpg"
img2 = "/home/timothee/Documents/programmation/DIHT/data/original_data/002/IMG_20240220_225113575.jpg"
f1 = sift.get_features(img1)
f2 = sift.get_features(img2)

In [3]:
sift.compute_distance(f1, f2)

0.002506265664160401